# Iterative PTO on the open stack (Exp4)

Preference trees + K-turn look-ahead + oracle scoring -> a `(chosen, rejected)` pair per branch
point -> **DPO**. Each iteration regenerates its own data from the *current* policy, and those same
conversations are the eval set for the policy that produced them.

Every LLM role except the therapist is an **open model served by one local vLLM
OpenAI-compatible endpoint**, so a full arm costs **$0 in API**. The therapist is the same
Llama-3.2-1B + LoRA as Exp3 -- deliberately, because the policy is not the variable here.

## The cell order is a contract, not a style

| # | cell | why it must be here |
|---|---|---|
| 0 | Colab bootstrap + install | nothing may import torch yet |
| 1 | flat globals | the only place a number is typed |
| 2 | runtime detect + auth | the open stack needs no vendor key at all |
| 3 | **`serve_roles()`** | vLLM's `gpu_memory_utilization` is a **pre-allocation**; the server must claim it BEFORE the trainer starts claiming the spiky remainder |
| 4 | oracle sanity `--quick` | an open grader can honour the schema perfectly and still be degenerate |
| 5 | `import trl` **then** torch | on the local sm_120 card the reverse order segfaults at CUDA init (exit 139) |
| 6 | build + validate config | `EXPERIMENT_NAME` is **computed** here, never typed |
| 7 | tokenizer / personas / base policy / resume | |
| 8 | the visible orchestration loop | the science should be readable in the notebook, not buried in a module |
| 9 | final eval generation | `N` iterations produce `N+1` conversation folders |
| 10 | inspection | TensorBoard + the cross-iteration dashboard |

## Five things this notebook will let you get wrong if you stop reading

1. **`greedy` mode requires an EVEN `MIN_CONV_LENGTH`.** The trunk seed is *sliced* off the step-1
   conversation and has to end on a **patient** turn, so the trunk's first branch point is a
   therapist turn. `validate_config` refuses an odd value; an odd one that slipped through would
   not fail, it would seed every trunk one turn out of phase.
2. **`DPO_BETA` is the DPO loss temperature, NOT GRPO's KL beta.** They are different quantities in
   different losses. Do not "match" them across the two notebooks.
3. **`iteration_N/pref_pairs/pairs.csv` is BOTH the audit trail AND the completion marker.** The
   preference build is PTO's dominant phase (5.7 of 8.1 hours in the Exp3 measurement), so its
   presence makes a resumed iteration reload the pairs and skip the build. An **EMPTY** `pairs.csv`
   therefore reloads 0 pairs, skips the build, and fails the zero-pairs guard. The fix is to
   **DELETE the file** and rebuild. It is *never* to lower `PREF_FILTER_TAU`: tau is not encoded in
   `EXPERIMENT_NAME`, so changing it mid-arm writes two different configurations into one folder
   with nothing on disk able to tell them apart.
4. **`TRAIN_BATCH_SIZE = 2` is a memory constraint, not a throughput choice.** DPO materialises
   full-**sequence** LM-head logits over a 128k vocab for four forward passes
   (chosen/rejected x policy/reference). Raise `GRADIENT_ACCUMULATION_STEPS` instead: 2 x 8 = 16
   pairs per optimizer step, matched to GRPO's 16 prompts per step.
5. **PTO is the framework; DPO is the loss.** PTO's data is preference pairs; GRPO's data is
   prompts and has no preferences in it. Never call GRPO's prompts "pref data", and never call
   PTO's pairs "GRPO data".

> Exp3 and Exp4 scores are **not** on the same axis (different grader). Compare within Exp4 only.

---
## 0. Colab bootstrap + environment

Mount Drive, `cd` into `code/pto/`, put `code/` on `sys.path`, then bring the environment up.

The install cell is **self-installing and guarded** — just run it. On a fresh Colab runtime it
installs the stack and asks you to restart; on a runtime that is already correct it prints one
line and moves on. Nothing to uncomment. It decides by reading installed package **metadata**
(never importing anything), and it refuses to install when the environment already matches, when
`AUTO_INSTALL = False`, or when you are **not on Colab** — where it would write vLLM and a CUDA
torch into the repo's `.venv`.

**Order matters when it does install:** vLLM first (it pins its own torch), pinned training stack
on top. vLLM is unpinned, but the cell warns below **0.19.1**, the floor for Gemma 4 support.

Nothing in this section may import torch: the vLLM server in section 3 has to carve out its
pre-allocation first, and on the local Blackwell card `import trl` must precede torch (section 5).

In [ ]:
# Colab: mount Drive + cd into code/pto/. Local: only the sys.path work runs.
# NO torch here -- see the section header.
import os
import sys

if "google.colab" in sys.modules or os.environ.get("COLAB_RELEASE_TAG"):
    from google.colab import drive

    drive.mount("/content/drive")
    _colab_dir = "/content/drive/MyDrive/Thesis_PTO_GRPO/Exp4_OpenStack/code/pto"
    if os.path.isdir(_colab_dir):
        os.chdir(_colab_dir)
    else:
        print(f"WARNING: Colab code dir does not exist: {_colab_dir}")

CODE_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
for _p in (CODE_DIR, os.getcwd()):
    if _p not in sys.path:
        sys.path.insert(0, _p)

print("cwd:              ", os.getcwd())
print("code/ on sys.path:", CODE_DIR)

In [ ]:
# =============================================================================
#  CELL 0 -- environment. This cell RUNS ITSELF; there is nothing to uncomment.
# =============================================================================
# GUARDED, not commented out. Three things it refuses to do:
#
#   * reinstall on a runtime that is already correct -- minutes of churn, and
#     re-installing vLLM re-resolves torch underneath a stack that was fine;
#   * install anything OUTSIDE Colab -- this notebook is importable locally for
#     smoke tests, and a vLLM/CUDA install into the repo's .venv is not
#     something to do by accident;
#   * install in the wrong order -- vLLM brings its own torch wheel, so it goes
#     FIRST and the pinned stack is layered on top. Installing vLLM last
#     silently replaces the torch the training stack was resolved against.
#
# So: a fresh runtime installs once and asks for a restart; every later run
# prints one line and moves on.
# ⚠ Byte-identical to the GRPO notebook's cell 0 on purpose -- the two install
# cells drifting apart is exactly how one method ends up on a different stack.
AUTO_INSTALL = True        # False -> report what is missing, change nothing

import os
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version

IS_COLAB = "google.colab" in sys.modules or bool(os.environ.get("COLAB_RELEASE_TAG"))

# The repo's validated set (requirements.txt). torch is deliberately NOT pinned
# here: Colab ships a CUDA build and vLLM pins its own on top of it.
PINNED = {
    "accelerate": "1.13.0",
    "datasets": "4.8.5",
    "huggingface_hub": "1.14.0",
    "numpy": "2.4.4",
    "openai": "2.36.0",
    "pandas": "3.0.3",
    "peft": "0.19.1",
    "safetensors": "0.7.0",
    "tensorboard": "2.20.0",
    "transformers": "5.8.1",
    "trl": "1.4.0",
}
# vLLM is deliberately UNPINNED (requirements.txt carries no validated version),
# but Gemma 4 needs at least this -- per the official vLLM Gemma 4 recipe. An
# older build is treated as work to do, not as a warning to act on by hand:
# it cannot serve the grader at all, so section 3 would fail either way.
VLLM_MIN = (0, 19, 1)


def _v(pkg):
    """Installed version string, or None. Reads package METADATA -- imports nothing.

    That matters here: this cell must not pull torch in, both for the sm_120
    import-order contract and because section 3 has to start vLLM first.
    """
    try:
        return version(pkg)
    except PackageNotFoundError:
        return None


def _older_than(ver, minimum):
    """True iff *ver* parses as a release older than *minimum*. Unparseable -> False."""
    try:
        return tuple(int(x) for x in ver.split("+")[0].split(".")[:3]) < minimum
    except (AttributeError, ValueError):
        return False           # a dev/nightly build -- do not cry wolf


_off_pin = {p: (_v(p), want) for p, want in PINNED.items() if _v(p) != want}
_vllm = _v("vllm")
_vllm_bad = _vllm is None or _older_than(_vllm, VLLM_MIN)
# Colab pre-bakes torchao < 0.16.0, and peft 0.19.1 does not merely ignore it:
# get_peft_model's LoRA dispatcher calls dispatch_torchao, which RAISES against
# that version, so attaching the adapter fails outright. Nothing here uses it.
_torchao = _v("torchao")
_work = bool(_off_pin) or _vllm_bad or _torchao is not None


def _report():
    for _p, (_have, _want) in sorted(_off_pin.items()):
        print(f"  {_p}: have {_have}, want {_want}")
    if _vllm is None:
        print("  vllm: not installed")
    elif _older_than(_vllm, VLLM_MIN):
        print(f"  vllm {_vllm}: older than {'.'.join(map(str, VLLM_MIN))}, the Gemma 4 floor")
    if _torchao is not None:
        print(f"  torchao {_torchao}: installed, and peft 0.19.1 raises against it")


if not _work:
    print(f"environment OK -- {len(PINNED)} pinned packages match, "
          f"vllm {_vllm}, torchao absent")
elif not AUTO_INSTALL:
    print("AUTO_INSTALL=False -- nothing was changed. Outstanding:")
    _report()
elif not IS_COLAB:
    print("NOT Colab -- refusing to install (this would write into the local .venv).")
    print("Outstanding, for reference:")
    _report()
else:
    def _pip(*args):
        print(f"  $ pip {' '.join(args)}")
        subprocess.check_call([sys.executable, "-m", "pip", *args])

    if _vllm_bad:
        print(f"{'installing' if _vllm is None else f'upgrading vLLM {_vllm} ->'} vLLM FIRST "
              f"(it pins its own torch) -- takes several minutes")
        _pip("install", "-q", "-U", "vllm")
    if _off_pin or _vllm_bad:
        # The FULL list, not just the off-pin subset: a fresh vLLM may have moved
        # numpy/transformers underneath it, so re-assert the whole validated set.
        print("pinning the training stack on top of vLLM's torch")
        _pip("install", "-q", *[f"{p}=={v}" for p, v in PINNED.items()])
    if _torchao is not None:
        print(f"removing torchao {_torchao} (peft 0.19.1 raises inside its LoRA dispatcher)")
        _pip("uninstall", "-y", "-q", "torchao")

    print("\n" + "=" * 74)
    print("  RESTART THE RUNTIME NOW  (Runtime -> Restart session), then run this")
    print("  cell again -- it will print one line and skip. Installing over modules")
    print("  the kernel has already imported leaves it holding the old ones.")
    print("=" * 74)

# bitsandbytes is only needed for USE_4BIT=True, which Exp4 never runs (4-bit induced ~30x
# more phrase-loop degeneration on this same base model in Exp2, which moved the whole score
# axis). If you ever flip that toggle: pip install bitsandbytes==0.49.2

---
## 1. Configuration -- the only place a number is typed

`EXPERIMENT_NAME` is **not** here. `core.config.build_pto_config` computes it from the values below
via `naming.build_experiment_name`:

```
PTO4_{QTAG}_LA{K}_MCL{N}_M{M}_PT{greedy|indep}_O{oracle_tag}_Pat{patient_tag}
```

Exp3 typed that string by hand, so changing `ORACLE_MODEL_ID` wrote a differently-rewarded policy
into the default arm's folder. Here the name is derived from the config that is about to be frozen,
and the failure is structurally impossible.

**The knobs the arm name does NOT encode** -- `PREF_FILTER_TAU`, every temperature, the learning
rate, the batch sizes, `LOOKAHEAD_SUB_BATCH_SIZE`, `GREEDY_TRUNK_TARGET_LEN`, `NUM_ITERATIONS` --
change the run without changing the folder. `run_metadata.json` plus its append-only
`run_metadata_history.jsonl` are the only record that can tell two such runs apart, which is why
changing one of them mid-arm is a science change with no trace in the layout.

In [ ]:
# =============================================================================
#  CELL 1 -- flat globals. Everything downstream reads a frozen config, never a global.
# =============================================================================

# --- runtime profile ---------------------------------------------------------
RUN_MODE   = "full"          # audit-only label. Exp4 has NO mode_tag path level.
QUICK_TEST = False           # see the override block at the bottom of this cell
SEED       = 42

# --- therapist policy (fixed across every Exp4 arm; not encoded in the arm name) ---
BASE_MODEL_ID = "meta-llama/Llama-3.2-1B"
TOKENIZER_ID  = "meta-llama/Llama-3.2-1B"
USE_4BIT      = False        # bf16. 4-bit induced ~30x more phrase-loop degeneration in Exp2, which
                             # the oracle floors -- it moves the whole score axis for nothing gained
                             # on a 1B model.

# --- roles ------------------------------------------------------------------
# provider: "openai_compat" (a local vLLM server) | "openai" | "anthropic".
# base_url stays None for openai_compat: serve_roles() fills in the port it actually assigned.
# request_timeout is PER ATTEMPT (the Exp3 defect this fixes): the safe shape is a SHORT
# per-attempt bound x MANY retries, never a long total budget. Exhausting a long budget freezes a
# simulation, and one frozen sim shifts the mean AND the std of its whole group.
#
# MODEL CHOICE (both ungated, Apache 2.0; sizes read off the HF API 2026-08-26):
#   google/gemma-4-E4B-it  7.996B params, 14.89 GiB bf16  <- default: the grader IS the instrument,
#                                                            and the E4B has better odds of passing
#                                                            the sanity gate's rank-agreement bar
#   google/gemma-4-E2B-it  5.123B params,  9.54 GiB bf16  <- fallback if E4B is too slow/tight;
#                                                            drop VLLM_GPU_MEMORY_UTILIZATION to 0.35
# Switching model = edit the three ids below; the arm name picks up the new tag automatically.
# Run the FULL oracle_sanity against BOTH before committing an arm -- choose by Spearman + spread,
# not by size.
ORACLE_MODEL_ID         = "google/gemma-4-E4B-it"
ORACLE_PROVIDER         = "openai_compat"
ORACLE_BASE_URL         = None
ORACLE_DISABLE_THINKING = True      # Gemma 4 must be told per request, or it spends the budget thinking
ORACLE_REQUEST_TIMEOUT  = 120.0     # a scoring call emits a whole JSON rubric
ORACLE_MAX_RETRIES      = 3

PATIENT_MODEL_ID         = "google/gemma-4-E4B-it"
PATIENT_PROVIDER         = "openai_compat"
PATIENT_BASE_URL         = None
PATIENT_DISABLE_THINKING = True
PATIENT_REQUEST_TIMEOUT  = 90.0     # one utterance; keep it <= ORACLE_REQUEST_TIMEOUT (see section 6)
PATIENT_MAX_RETRIES      = 8

# The judge never runs in this notebook -- it grades afterwards, in the EDA, and partitions the score
# lake as judge=<tag>/. It is configured here so run_metadata records it and so plan_servers can see
# that all three roles are the same model and start ONE server.
JUDGE_MODEL_ID         = "google/gemma-4-E4B-it"
JUDGE_PROVIDER         = "openai_compat"
JUDGE_BASE_URL         = None
JUDGE_DISABLE_THINKING = True
JUDGE_REQUEST_TIMEOUT  = 120.0
JUDGE_MAX_RETRIES      = 3

# --- iterative loop (matched to the GRPO notebook) ---------------------------
NUM_ITERATIONS             = 6
# 1 on purpose, matched across BOTH methods -- and 1 is not the same thing in the two trainers,
# which is exactly why 2 was wrong: a GRPO epoch re-SAMPLES G fresh completions per prompt and
# re-grades them (2 epochs = twice the reward-side work, on partially-updated weights), while a DPO
# epoch re-treads the SAME fixed pairs. epochs=1 makes "one pass over data produced by this
# iteration's policy" true for both methods; raise NUM_ITERATIONS, not this, for more updates.
EPOCHS_PER_ITERATION       = 1
NUM_CONVERSATIONS_PER_ITER = 96     # one per patient persona (the full V3 permutation set)
NUM_UTTERANCES_FOR_DATA    = 49     # target conversation length, therapist + patient combined

# MIN_CONV_LENGTH -- the training-context floor, in utterances (therapist + patient combined).
# Rank agreement between a short cut and the full-conversation score is barely above chance at 2
# utterances and only clears 0.8 at ~10, so training on 2-utterance slices optimises "did the
# opening look promising?" rather than "did the session deliver?".
#   greedy mode:      where the trunk STARTS -- must be EVEN so the sliced seed ends on a patient turn.
#   independent mode: a branch-point filter.
MIN_CONV_LENGTH            = 12

# --- conversation generation -------------------------------------------------
CONVERSATION_BATCH_SIZE          = 64     # concurrent simulations. On the 12 GB local card this is a
                                          # SAFETY setting: ~2.6 GB weights + ~1.1 GB per conversation,
                                          # and an over-budget request REBOOTS the machine.
TEMPERATURE_THERAPIST_GEN        = 0.9
TEMPERATURE_PATIENT              = 0.7
MAX_TOKENS_PER_RESPONSE          = 200
THERAPIST_MAX_INPUT_TOKENS       = 2048
PATIENT_CONCURRENCY              = 96     # in-flight patient calls; shared with the look-ahead rollout
MAX_GEN_RETRIES_WITHOUT_PROGRESS = 3
GEN_VERBOSE                      = True
GEN_VERBOSE_DETAILED             = False

# --- prompt + completion limits ---------------------------------------------
STOP_STRINGS              = ["<|im_end|>", "<|im_start|>"]   # the therapist is a BASE model with a
                                                             # hand-written ChatML template; the markers
                                                             # are ordinary BPE pieces, so without these
                                                             # it self-plays both speakers
MAX_ALLOWED_PROMPT_LENGTH = 2048
MAX_COMPLETION_LENGTH     = 200

# --- oracle reward -----------------------------------------------------------
# Training reward = Q1 + Q2, unweighted mean across questionnaires (so Q1's 5 items and Q2's 17 carry
# equal weight). QUESTIONNAIRE_IDS drives the QTAG in the arm name.
QUESTIONNAIRE_IDS        = [1, 2]
EVAL_TEMPERATURE         = 0.0    # > 0 makes the grader a random variable and a re-score irreproducible
ORACLE_MAX_TOKENS        = 256
ORACLE_MAX_RETRIES       = 3
ORACLE_REQUEST_TIMEOUT   = 120.0
ORACLE_MAX_CONCURRENCY   = 64
ORACLE_MIN_SUCCESS_RATIO = 0.5    # below this the batch RAISES: training on a biased subset of the
                                  # candidates is worse than stopping

# --- look-ahead: simulate K extra turns before the oracle reads the transcript ---
# K is the lever the whole experiment exists to measure; the sweep is K in {0, 5} on BOTH methods,
# which is what isolates look-ahead from the loss family. K=0 disables it entirely.
LOOKAHEAD_K              = 0
LOOKAHEAD_TEMP_THERAPIST = 0.9
LOOKAHEAD_TEMP_PATIENT   = 0.7
LOOKAHEAD_MAX_TOKENS     = 200
LOOKAHEAD_MAX_INPUT_TOKENS = 2048
LOOKAHEAD_SUB_BATCH_SIZE = 64     # therapist generate cap per look-ahead turn. Auto-halved on OOM and
                                  # the halving is STICKY, so the realised value can differ from this
                                  # one -- it is recorded per iteration for exactly that reason.

# --- preference-tree construction (PTO-specific) -----------------------------
# greedy = true PTO: slice an MCL-length prefix off the step-1 conversation as the trunk seed, then at
# each therapist turn sample M completions -> look-ahead -> oracle -> APPEND THE BEST TO THE TRUNK, so
# the choice made at depth d is the context every branch at depth d+2 is sampled from.
# independent = branch a pre-recorded conversation and never feed the winner back (alternate arm,
# cheaper control; if it produces the same pairs, the feedback bought nothing).
PREF_TREE_MODE            = "greedy"    # "greedy" | "independent"; baked into EXPERIMENT_NAME
NUM_BRANCHES_PER_TURN     = 8           # M -- mirrors GRPO's G=8
PREF_FILTER_TAU           = 0.1         # emit a pair only when chosen - rejected > tau. NOT in the arm
                                        # name: changing it mid-arm mixes two configurations in one folder.
BRANCH_SAMPLE_TEMPERATURE = 1.2         # = GRPO_TEMPERATURE
BRANCH_MAX_TOKENS         = 200
GREEDY_TRUNK_TARGET_LEN   = None        # None = NUM_UTTERANCES_FOR_DATA. Lowering it is a speed lever
                                        # AND a science change (shallower context at every branch point),
                                        # and it is not in the arm name.

# --- DPO ---------------------------------------------------------------------
LEARNING_RATE               = 1e-5
DPO_BETA                    = 0.1        # the DPO loss temperature -- NOT GRPO's KL beta
DPO_LOSS_TYPE               = "sigmoid"  # "sigmoid" | "ipo" | ...
DPO_PRECOMPUTE_REF_LOGPS    = True       # reference log-probs in a no-grad pre-pass: semantically
                                         # identical (the reference is frozen) and frees its VRAM
DPO_GRADIENT_CHECKPOINTING  = True       # trades ~30% step time for activation memory; it does NOT
                                         # shrink the logits tensor, so it complements the batch size
TRAIN_BATCH_SIZE            = 2          # PAIRS per device -- the memory lever. Sizes the full-SEQUENCE
                                         # LM-head logits tensor over a 128k vocab. Do not raise it.
EVAL_BATCH_SIZE             = 4          # eval is no-grad, so a little more headroom
GRADIENT_ACCUMULATION_STEPS = 8          # 2 x 8 = 16 pairs/step, matched to GRPO's 16 prompts/step
WARMUP_STEPS_RATIO          = 0.01
EVAL_SPLIT_RATIO            = 0.05       # held out by CONVERSATION, not by pair: greedy's pairs share a
                                         # monotonically growing prefix, so a pair-level split would
                                         # measure memorisation

# --- LoRA --------------------------------------------------------------------
LORA_R              = 16
LORA_ALPHA          = 16
LORA_DROPOUT        = 0.05
LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj", "up_proj", "down_proj", "gate_proj"]

# --- checkpointing, logging, capture ----------------------------------------
LOGGING_STEPS    = 1
SAVE_STRATEGY    = "steps"
SAVE_STEPS       = 10
SAVE_TOTAL_LIMIT = 2          # >= 2 so resume can walk back over a torn newest write
REPORT_TO        = ["tensorboard"]        # Exp4 is TensorBoard only -- there is no W&B wiring
PUSH_TO_HUB      = False
HUB_ENTITY       = "LBK95"                # only used when PUSH_TO_HUB is True

SAVE_EDA_GENERATIONS       = True   # iteration_N/eda/generations.jsonl: one row per branch point, all M
                                    # candidates nested, with scores and look-ahead tails
SAVE_LOOKAHEAD_TRANSCRIPTS = True   # the per-candidate tails are the dominant size lever -- this switch
                                    # shrinks generations.jsonl AND the mid-build _progress.json
TB_LIVE_LOGGING            = False  # opt-in continuous run-level TB view at runs/<ARM>/tb_live/
TB_SAMPLE_COMPLETIONS_N    = 8

# --- vLLM serving (section 3) -----------------------------------------------
VLLM_BASE_PORT              = 8000
# A PRE-ALLOCATION, not a growing ceiling. Show the arithmetic (project rule):
#   E4B default: 0.50 x 40 GiB = 20 GiB = 14.89 GiB weights + ~4-5 GiB KV pool + vLLM overhead.
#   E2B fallback: use 0.35     = 14 GiB =  9.54 GiB weights + ~3.5-4 GiB KV pool.
# The old 0.25 default assumed "~3 GB" of Gemma weights; the real bf16 checkpoints are 14.89 /
# 9.54 GiB (HF API, 2026-08-26), and at 0.25 the E4B server cannot even hold its weights. Low-ish
# and started FIRST, because training memory is the spiky side; verify the measured weights line
# that section 3 prints before trusting any of this.
VLLM_GPU_MEMORY_UTILIZATION = 0.50
VLLM_MAX_MODEL_LEN          = 16384  # do NOT lower this to save memory -- see the section 3 warning
VLLM_DTYPE                  = "bfloat16"
VLLM_STARTUP_TIMEOUT        = 900.0
VLLM_EXTRA_ARGS             = []     # e.g. ["--enable-prefix-caching"] on a vLLM old enough to need it
VLLM_LOG_DIR                = None   # None -> /content/vllm_logs on Colab, ./_vllm_logs locally --
                                     # NEVER the Drive mount: a wedged FUSE write on the server's
                                     # stdout can stall the server itself

# --- oracle JSON-schema strictness ------------------------------------------
# False omits strict:true from the json_schema response format, for a pinned vLLM that 400s on the key.
# A grader that only passes with this False is being held to the schema by the SERVER rather than
# following the rubric. Recorded in run_metadata.json by write_run_metadata.
ORACLE_SCHEMA_STRICT = True

# --- paths -------------------------------------------------------------------
DATA_ROOT      = None   # None -> <workspace>/data (the three Google Drive symlinks)
WORKSPACE_ROOT = None   # None -> resolved by walking up for the Exp4_OpenStack root

# =============================================================================
#  Quick-test overrides
#  M is dropped to 3 ON PURPOSE: M is encoded in EXPERIMENT_NAME (_M3_), so a quicktest lands in a
#  DIFFERENT folder than the real arm. Exp4 has no mode_tag path level, so a smoke run that kept
#  every name-encoded axis would write straight into the arm you intend to run for real. (The EDA
#  also disambiguates display labels on collision and can pin an arm by experiment_name, so a
#  quicktest sibling can never silently merge into the real arm's figures.)
# =============================================================================
if QUICK_TEST:
    NUM_ITERATIONS              = 2
    NUM_CONVERSATIONS_PER_ITER  = 4
    NUM_UTTERANCES_FOR_DATA     = 16    # a trunk seeded at MCL=12 reaches this in ~2 therapist depths
    NUM_BRANCHES_PER_TURN       = 3     # -> _M3_ in the arm name -> a disjoint folder
    MAX_COMPLETION_LENGTH       = 64
    BRANCH_MAX_TOKENS           = 64
    GRADIENT_ACCUMULATION_STEPS = 1
    CONVERSATION_BATCH_SIZE     = 2
    ORACLE_MAX_CONCURRENCY      = 8
    PATIENT_CONCURRENCY         = 8
    LOOKAHEAD_SUB_BATCH_SIZE    = 8
    PUSH_TO_HUB                 = False
    RUN_MODE                    = "quicktest"
    print("QUICK_TEST: minimal settings, M=3 -> a separate arm folder")

print(f"cell 1 loaded: PTO K={LOOKAHEAD_K} MCL={MIN_CONV_LENGTH} M={NUM_BRANCHES_PER_TURN} "
      f"mode={PREF_TREE_MODE} tau={PREF_FILTER_TAU} | oracle={ORACLE_MODEL_ID} "
      f"patient={PATIENT_MODEL_ID}")
print(f"  loop: {NUM_ITERATIONS} iterations x {EPOCHS_PER_ITERATION} epoch(s), seed {SEED}")
print("EXPERIMENT_NAME is computed in section 6 -- do not type it here.")

---
## 2. Runtime, credentials, import-order guard

`authenticate` asks for a vendor key only when a role is actually bound to that vendor. On the
default all-open stack it asks for **none** -- that is the point of Exp4. The Hugging Face token is
still resolved because `meta-llama/Llama-3.2-1B` is gated, but a missing one only warns (the weights
may already be cached).

`assert_import_order` inspects `sys.modules` only. It never imports torch, so calling it cannot
create the condition it checks for.

In [ ]:
from core.runtime import (
    assert_import_order,
    authenticate,
    describe_environment,
    detect_host,
    format_environment,
    resolve_workspace_root,
)

assert_import_order()          # torch must not be imported before trl on the local sm_120 card

HOST      = detect_host()
WORKSPACE = resolve_workspace_root()

_providers = (ORACLE_PROVIDER, PATIENT_PROVIDER, JUDGE_PROVIDER)
AUTH = authenticate(
    hf=True,                                   # the therapist base is gated; a miss only warns
    openai="openai" in _providers,             # required=True when asked for: a missing key raises
    anthropic="anthropic" in _providers,
)

print(format_environment(describe_environment()))
print(f"host={HOST}  workspace={WORKSPACE}")
print(f"roles -> providers {_providers}")
if not any(p in ("openai", "anthropic") for p in _providers):
    print("all-open stack: no vendor API key is needed and none was requested ($0 API)")

---
## 3. Serve the open roles -- BEFORE any torch import

`plan_servers` **dedupes by model id**: oracle + patient + judge on the same Gemma is exactly
**one** vLLM server, because the roles differ only in per-request sampling parameters. Three servers
would triple the weight memory and split the prefix cache three ways -- and prefix caching is
precisely what makes the rubric-first oracle prompt cheap.

`serve_roles` is **idempotent**. A healthy server already on the port serving the right model is
adopted, not duplicated, so re-running this cell after a kernel restart is free. A port that answers
with a *different* model raises rather than adopting: silently talking to the wrong grader would
produce a complete, valid-looking, wrongly-scored arm.

> **`gpu_memory_utilization` is a pre-allocation, not a growing ceiling.** vLLM grabs the fraction at
> startup and keeps it. Start the server FIRST, because the trainer is the spiky side of the budget.
> The fraction is sized from the MEASURED checkpoint: Gemma-4-E4B-it is **14.89 GiB** bf16 (E2B:
> 9.54 GiB) -- so 0.50 of a 40 GB A100 = 20 GiB = weights + a ~4-5 GiB KV pool. **Check the
> weights line this cell prints against those numbers** -- the whole budget stands on it.

> **`VLLM_MAX_MODEL_LEN` must stay at 16384.** Measured against 192 real Exp3 transcripts, full Q2
> oracle prompts run to a maximum of 10,042 tokens; at 8192, 2.1% of Q2 prompts and 1.0% of Q1
> prompts would not fit. Those are the LONGEST conversations, session length varies by arm and by K,
> and an unscoreable conversation is simply *absent* rather than an error -- so the dropout would be
> an arm-dependent silent bias on the headline metric. The memory cost of the higher cap is near
> zero (the KV pool is sized by `gpu_memory_utilization`; `max_model_len` caps one sequence). Give
> memory back through `VLLM_GPU_MEMORY_UTILIZATION` instead.

In [ ]:
from roles import make_binding
from tools.vllm_serve import report_weights_gib, serve_roles

_ROLE_TEMPLATES = {
    "oracle": (ORACLE_PROVIDER, ORACLE_MODEL_ID, ORACLE_BASE_URL, ORACLE_DISABLE_THINKING,
               ORACLE_REQUEST_TIMEOUT, ORACLE_MAX_RETRIES),
    "patient": (PATIENT_PROVIDER, PATIENT_MODEL_ID, PATIENT_BASE_URL, PATIENT_DISABLE_THINKING,
                PATIENT_REQUEST_TIMEOUT, PATIENT_MAX_RETRIES),
    "judge": (JUDGE_PROVIDER, JUDGE_MODEL_ID, JUDGE_BASE_URL, JUDGE_DISABLE_THINKING,
              JUDGE_REQUEST_TIMEOUT, JUDGE_MAX_RETRIES),
}
_unserved = {
    role: make_binding(provider, model, base_url=base_url, disable_thinking=no_think,
                       request_timeout=timeout, max_retries=retries)
    for role, (provider, model, base_url, no_think, timeout, retries) in _ROLE_TEMPLATES.items()
}

# report_weights_gib reads the launch log, and an ADOPTED handle only finds one if the same log_dir
# was used by whoever launched it -- so keep this path stable across re-runs of the cell.
VLLM_LOGS = VLLM_LOG_DIR or ("/content/vllm_logs" if HOST == "colab"
                             else os.path.join(os.getcwd(), "_vllm_logs"))

# ROLE_BINDINGS is what build_pto_config reads (it takes precedence over the *_MODEL_ID globals),
# because only these carry the base_url of the port that was actually assigned.
ROLE_BINDINGS, SERVER_HANDLES = serve_roles(
    _unserved,
    base_port=VLLM_BASE_PORT,
    log_dir=VLLM_LOGS,
    timeout=VLLM_STARTUP_TIMEOUT,
    gpu_memory_utilization=VLLM_GPU_MEMORY_UTILIZATION,
    max_model_len=VLLM_MAX_MODEL_LEN,
    dtype=VLLM_DTYPE,
    extra_args=tuple(VLLM_EXTRA_ARGS),
)

print(f"\n{len(SERVER_HANDLES)} vLLM server(s) for {len(ROLE_BINDINGS)} role(s):")
for _model, _handle in SERVER_HANDLES.items():
    # Phase 1 gate: the budget expects ~14.89 GiB for E4B (~9.54 for E2B), measured off the HF
    # API. Check the MEASURED figure from the startup log against that -- a big mismatch means
    # the serving stack is not loading what the arithmetic assumed (offload? quantized variant?).
    # "unknown" means the log line was reworded, not that it failed.
    _gib = report_weights_gib(_handle)
    print(f"  {_model:<28} {_handle.base_url}   weights "
          f"{('%.2f GiB' % _gib) if _gib is not None else 'unknown'}")

---
## 4. Oracle sanity gate (`--quick`) -- run this before spending a GPU-hour

An open-weights grader fails in two ways and only one of them is loud:

1. it ignores the schema or returns the wrong number of item scores -- caught by the validation
   ladder, and surfaces as retries and then as **biased missingness**;
2. it honours the schema perfectly and returns **degenerate** scores (every item a 4, near-zero
   variance across conversations). That parses, writes valid parquet, and produces a grader that
   cannot tell any two arms apart. Nothing downstream flags it: the contrast tables just come back
   near zero and look like a finding.

The gate scores a committed fixture of real Exp3 transcripts spanning Q1+Q2 1.00 to 5.00, with their
frozen `gpt-4o-mini` scores as a reference. **Hard gates** (they block): perfect schema validity, and
non-degenerate spread. **Soft** (reported only): Spearman rank agreement and the level offset --
Exp3 and Exp4 are not on the same score axis, so an offset is expected and uninformative.

`--quick` scores only 2 transcripts, so the spread gate is genuinely weaker here. This is the
pre-flight; the **full** report is the Phase 2 gate before any real arm. The binding used is the one
the trainer will use, because testing a different binding proves nothing about the run.

In [ ]:
from core.concurrency import run_async
from core.oracle import set_openai_compat_strict
from tools.oracle_sanity import check_gates, format_report, run_sanity, write_report

# Module-level flag, so this also governs the trainer's own oracle calls for the rest of the process.
set_openai_compat_strict(ORACLE_SCHEMA_STRICT)

SANITY_REPORT = run_async(run_sanity(
    ROLE_BINDINGS["oracle"],
    questionnaire_ids=QUESTIONNAIRE_IDS,
    # quick=True samples 2 transcripts instead of 12. That existed to save oracle CALLS -- and on
    # the default open stack a call costs nothing, so the saving buys nothing while giving up the
    # only gate that matters: the spread check needs n >= MIN_N_FOR_SPREAD_GATE to tell a
    # template-answering grader apart from two conversations that happened to score alike, and
    # below that it is reported rather than enforced. So: full gate whenever the grader is local,
    # quick only when someone is paying per call.
    quick=(ORACLE_PROVIDER != "openai_compat"),
    concurrency=min(8, ORACLE_MAX_CONCURRENCY),
    max_tokens=ORACLE_MAX_TOKENS,        # keep EQUAL to the trainer's budget: a gate run at a larger
                                         # budget tests a different configuration than the run
    max_retries=ORACLE_MAX_RETRIES,
    request_timeout=ORACLE_REQUEST_TIMEOUT,
))
print(format_report(SANITY_REPORT))

_passed, _reasons = check_gates(SANITY_REPORT)
if not _passed:
    raise RuntimeError(
        "oracle sanity HARD GATE failed -- do not train against this grader:\n  - "
        + "\n  - ".join(_reasons)
        + "\nFix the model, the prompt or the serving config. A degenerate grader still produces a "
          "complete arm; it just cannot tell any two arms apart."
    )
print("oracle sanity: hard gates PASSED (the report is archived next to run_metadata.json below)")

---
## 5. Imports -- `trl` BEFORE torch

On the local Blackwell card (sm_120) `from trl import ...` *after* torch is already imported
**segfaults at CUDA init** -- exit 139, no traceback, nothing to catch. Colab is unaffected, but the
order is kept everywhere so the notebook is runnable in both places.

In [ ]:
assert_import_order()

# trl FIRST -- and the CONCRETE import is the one that matters. trl 1.4.0's top level is a
# _LazyModule: after a bare `import trl` nothing of trl.trainer has actually loaded, so the native
# init would otherwise happen inside `from pto.pto_trainer import ...` BELOW -- i.e. AFTER torch --
# which is exactly the measured 3/3 segfault sequence on the local sm_120 card (exit 139, no
# traceback). Pulling the symbols here is what actually satisfies the ordering; assert_import_order
# cannot see this case because the lazy shell already puts "trl" in sys.modules.
import trl  # noqa: F401
from trl import DPOConfig, DPOTrainer  # noqa: F401

import gc
import random

# datasets BEFORE torch -- the second native-init pair. MEASURED on the local sm_120 card:
# importing torch first makes the datasets import an access violation (exit 139) inside
# pyarrow.dataset. Harmless-looking line order, unrecoverable crash. See CLAUDE.md.
import datasets  # noqa: F401

import torch

from core.concurrency import AsyncPrimitives
from core.config import build_pto_config
from core.lookahead import LookaheadState
from core.policy import (
    compute_cumulative_step_offset,
    list_iteration_checkpoints,
    resolve_start_state,
    setup_base_model,
    setup_tokenizer,
    sync_pad_token,
    vram_report,
)
from core.tb import RunTBLogger, plot_iteration_metrics, scan_scalar_tags
from core.timing import cumulative_seconds
from pto.pto_trainer import build_lora_config, run_final_eval, run_one_iteration, write_run_metadata
from roles import make_client, reset_client_cache
from tools.vllm_serve import ensure_alive

print(f"trl {trl.__version__} | torch {torch.__version__} | cuda={torch.cuda.is_available()}")

---
## 6. Freeze + validate the config

`build_pto_config(globals())` reads the flat globals once, freezes them into typed dataclasses,
**computes** `EXPERIMENT_NAME`, derives every path from it, and runs `validate_config` over the whole
bundle. It also warns about ALL-CAPS globals it never read that look like typos of ones it did --
because a misspelled knob (`LOOKAHED_K = 5`) is otherwise completely silent: the default is used, the
arm name says `LA0`, and the run is a different experiment than the one you configured.

`EXPERIMENT_NAME` is assigned *after* the build purely so later cells can print it. Re-running this
cell after changing K in cell 1 prints a warning that the stale value is ignored, then overwrites it
-- which is the intended behaviour, not a problem.

**One client for two roles.** `pto_trainer` takes a single `client` and hands it to both the patient
rollout and the oracle, so this notebook builds it from the **oracle** binding: the oracle's
per-attempt timeout is the larger of the two, so neither role's `asyncio.wait_for` is undercut by the
SDK's socket bound. That only works while both roles sit behind the same endpoint, which
`validate_config` enforces for both trainers (`core.config._roles_errors`).

In [ ]:
train_cfg, roles_cfg, gen_cfg, oracle_cfg, la_cfg, paths = build_pto_config(globals())

# Echo only -- the name was COMPUTED above and is never read back out of the globals.
EXPERIMENT_NAME = train_cfg.experiment_name

# One client object serves both the oracle and the look-ahead patient in this trainer, so a split
# stack (oracle on a vendor API, patient on vLLM) is not expressible here: the requests would all go
# to whichever endpoint the client was built from. build_pto_config REFUSED that above --
# core.config._roles_errors owns the rule for BOTH trainers, so the two notebooks cannot drift.

paths.ensure_run_dir()
print("oracle sanity report ->", write_report(SANITY_REPORT, paths.run_dir))

primitives = AsyncPrimitives(
    oracle_concurrency=oracle_cfg.max_concurrency,
    patient_concurrency=gen_cfg.patient_concurrency,
)
client = make_client(roles_cfg.oracle)

# ONE LookaheadState for the whole arm: it carries the sub-batch that OOM halving arrived at, and
# without it the OOM is re-paid at every depth of every trunk (~20 depths per iteration).
lookahead_state = LookaheadState()

# Only used on a fresh start, where resolve_start_state hands back a BARE base model with no adapter
# to load; ignored on every resume and every iteration after the first.
lora_config = build_lora_config(train_cfg)

print(f"\narm        {EXPERIMENT_NAME}")
print(f"run dir    {paths.run_dir}")
print(f"conv dir   {paths.conv_root}")
print(f"pairs/step {train_cfg.pairs_per_step}  (per_device {train_cfg.train_batch_size} x gas "
      f"{train_cfg.gradient_accumulation_steps})")

---
## 7. Tokenizer, personas, base policy

`generate_all_permutations` picks the therapist's *name* at random, so `random.seed(SEED)` is
re-applied immediately before the call: a resumed process must rebuild the **same** therapist system
prompt, or the arm's second half runs under a different prompt than its first. The 96 personas
themselves are deterministic (nested loops), and their index **is** the persona id -- `pers07.csv` is
persona 7 in every iteration, forever.

The tokenizer's ChatML template is overwritten unconditionally: it is part of the experiment
definition, not of the checkpoint.

In [ ]:
from system_prompts_builder import generate_all_permutations
from tools.generate_convs import therapist_prompt_pair

tokenizer = setup_tokenizer(train_cfg.tokenizer_id)
print(f"tokenizer {train_cfg.tokenizer_id}  vocab={len(tokenizer)}  pad={tokenizer.pad_token!r}")

random.seed(train_cfg.seed)          # fixes the therapist NAME across processes -- see the header
all_permutations = generate_all_permutations(only_expert_therapist=True)
therapist_system_prompt, therapist_init_utterance = therapist_prompt_pair(all_permutations)
if gen_cfg.num_conversations_per_iter > len(all_permutations):
    raise ValueError(
        f"NUM_CONVERSATIONS_PER_ITER={gen_cfg.num_conversations_per_iter} exceeds the "
        f"{len(all_permutations)} available personas"
    )
print(f"personas  {len(all_permutations)}  (using {gen_cfg.num_conversations_per_iter}/iteration; "
      f"the shuffle sets processing ORDER only -- files are named pers<ID>.csv)")

base_policy = setup_base_model(train_cfg.base_model_id, use_4bit=train_cfg.use_4bit)
sync_pad_token(base_policy, tokenizer)
print(f"vram after base load: {vram_report()}")

_existing = list_iteration_checkpoints(paths.run_dir)
print(f"existing iterations: {[n for n, _ in _existing] or 'none (fresh arm)'}")

---
## 8. Iterative PTO training -- the visible orchestration

Per iteration `n`:

1. **generate** 96 conversations with the policy the iteration STARTS with -- they are saved as
   `model_iter_{n-1}` and double as that policy's eval set;
2. **build preference pairs** (slice trunk seeds -> branch M -> look-ahead -> oracle -> tau filter ->
   append the best to the trunk). This is the dominant phase;
3. **DPO** on the pairs;
4. save `iteration_n/adapter/` -- and "iteration done" means the adapter's FILES are present, not
   just the directory (a torn save is treated as incomplete and resumed past).

**Resume is free at three levels.** Conversations resume per persona from disk; the preference build
resumes from `pref_pairs/_progress.json` (discarded, never merged, if its config fingerprint --
which includes tau -- differs; the finished `pairs.csv` carries the same fingerprint in a sidecar
and warns on mismatch); the DPO step resumes from the latest valid HF checkpoint -- and the policy
handed to the resumed trainer carries the ITERATION-START weights, so the DPO reference stays the
iteration start rather than silently becoming the crash checkpoint. **Each phase logs its own
timing line the moment it completes** (generation, build, training), so a killed process still
leaves its finished phases on the cost record; a reloaded build logs nothing extra -- the session
that built the pairs already recorded the time.

> If an iteration raises **"0 preference pairs"**, DELETE `iteration_N/pref_pairs/pairs.csv` and
> re-run. Do not lower `PREF_FILTER_TAU` -- tau is not in `EXPERIMENT_NAME`, so that silently writes
> two different configurations into one folder.

> `result.policy` must be rebound into `policy`: TRL hands back a possibly NEW wrapper, and reusing
> the object passed in would train iteration `n+1` from a stale one.

> ⚠ If you **Interrupt** this cell during the build, RESTART THE KERNEL before re-running it. The
> build runs on a daemon worker thread that an interrupt does not stop; re-running the cell on the
> live kernel starts a SECOND build against the same GPU and the same `_progress.json`.

In [ ]:
# --- pre-loop ----------------------------------------------------------------
# Once per process, BEFORE iteration 1: the current file is overwritten and the superseded payload is
# appended to run_metadata_history.jsonl, so a resume under changed knobs no longer erases what the
# earlier iterations actually ran under.
write_run_metadata(train_cfg, roles_cfg, gen_cfg, oracle_cfg, la_cfg, paths)

tb_logger = RunTBLogger(paths.run_dir, enabled=train_cfg.tb_live_logging)

start_iteration, policy, resume_checkpoint = resolve_start_state(paths.run_dir, base_policy, tokenizer)
cumulative_step_offset = compute_cumulative_step_offset(paths.run_dir)
print(f"\nstarting at iteration {start_iteration}/{train_cfg.num_iterations} "
      f"(cumulative step offset {cumulative_step_offset})")
print("=" * 78)

# Each iteration builds a fresh DPOTrainer: LoRA weights carry over, Adam state resets (warm restart).
iterations_run = []

for iteration in range(start_iteration, train_cfg.num_iterations + 1):
    # Phase boundary: probe the server(s). A restart invalidates every cached client -- the cached one
    # holds a connection pool to a process that no longer exists, and the symptom is a burst of
    # connection errors on the next phase, blamed on the wrong thing.
    _restarted = 0
    for _handle in SERVER_HANDLES.values():
        _before = _handle.restarts
        ensure_alive(_handle)
        _restarted += int(_handle.restarts > _before)
    if _restarted:
        reset_client_cache()
        client = make_client(roles_cfg.oracle)
        print(f"  {_restarted} vLLM server(s) restarted; rebuilt the client")

    result = run_one_iteration(
        iteration=iteration,
        start_iteration=start_iteration,
        resume_checkpoint=resume_checkpoint,
        cumulative_step_offset=cumulative_step_offset,
        policy=policy,
        tokenizer=tokenizer,
        client=client,
        permutations=all_permutations,          # the FULL list, indexed by persona_id
        sp_therapist=therapist_system_prompt,
        therapist_init_utterance=therapist_init_utterance,
        train_cfg=train_cfg,
        gen_cfg=gen_cfg,
        oracle_cfg=oracle_cfg,
        la_cfg=la_cfg,
        paths=paths,
        primitives=primitives,
        patient_binding=roles_cfg.patient,
        lora_config=lora_config,
        tb_logger=tb_logger,
        lookahead_state=lookahead_state,
    )

    policy = result.policy                      # TRL may hand back a NEW wrapper -- rebind
    cumulative_step_offset += result.step_delta
    resume_checkpoint = None                    # it describes THIS process's first iteration only
    iterations_run.append(iteration)

    print(f"  iter {result.iteration}: {result.n_pref_pairs} pairs from "
          f"{result.n_conversations} conversations | build {result.pref_pair_s / 60:.1f} min"
          f"{' (reloaded)' if result.pref_pairs_reloaded else ''} | "
          f"train {result.training_s / 60:.1f} min | {result.step_delta} steps | "
          f"look-ahead sub-batch {result.lookahead_sub_batch}")

    gc.collect()
    torch.cuda.empty_cache()

print(f"\n{len(iterations_run)} iteration(s) completed this process: {iterations_run}")

---
## 9. Final eval generation -- `model_iter_{NUM_ITERATIONS}`

Every trained iteration's conversations are generated by the policy it *starts* with, so `N`
iterations produce `model_iter_0 .. model_iter_{N-1}` and the last adapter would otherwise have no
eval data at all. This generate-only pass produces the `N+1`-th folder.

It is a separate cell so a loop that was interrupted can be resumed above and this run once, at the
end -- and it resumes per persona from disk, so a killed pass costs only what it had not yet written.
Its wall-clock is logged as `eval_gen_s` against the last iteration, which keeps the arm's total cost
the plain sum over `iteration_*/timing_sessions.jsonl`.

In [ ]:
final_conv_dir = run_final_eval(
    policy=policy,
    tokenizer=tokenizer,
    client=client,
    permutations=all_permutations,
    sp_therapist=therapist_system_prompt,
    therapist_init_utterance=therapist_init_utterance,
    train_cfg=train_cfg,
    gen_cfg=gen_cfg,
    paths=paths,
    primitives=primitives,
    patient_binding=roles_cfg.patient,          # must be the simulator the arm was TRAINED against
    la_cfg=la_cfg,
)

tb_logger.close()

print("\n" + "=" * 78)
print(f"PTO ARM COMPLETE  {EXPERIMENT_NAME}")
print("=" * 78)
print(f"  iterations   {train_cfg.num_iterations} "
      f"({len(iterations_run)} run in this process)")
print(f"  adapters     {[f'iteration_{n}' for n, _ in list_iteration_checkpoints(paths.run_dir)]}")
print(f"  run dir      {paths.run_dir}")
print(f"  conv dirs    {paths.conv_root}")
print(f"  final eval   {final_conv_dir}")

# n_sessions_production > 1 for an iteration means it was RESUMED, which means every per-PROCESS
# number recorded for it is wrong -- read the cumulative figures instead. That is the whole reason
# the per-phase log is append-only. Note it is the PRODUCTION count: run_final_eval below appends
# an eval-gen-only session to the last iteration of every healthy arm, so the raw session count
# would call every completed arm resumed. `production` is also the cost axis -- eval-gen MEASURES
# the final policy rather than producing it.
print("\n  per-iteration wall-clock (cumulative across sessions):")
for _n, _ in list_iteration_checkpoints(paths.run_dir):
    _t = cumulative_seconds(paths.iteration_dir(_n))
    print(f"    iteration_{_n}: production {_t.get('production_s', 0.0) / 3600:.2f} h "
          f"(generate {_t.get('generation_s', 0.0) / 3600:.2f}, "
          f"build {_t.get('pref_pair_s', 0.0) / 3600:.2f}, "
          f"train {_t.get('training_s', 0.0) / 3600:.2f}) "
          f"+ eval-gen {_t.get('eval_gen_s', 0.0) / 3600:.2f} h "
          f"over {int(_t.get('n_sessions_production', 0))}/"
          f"{int(_t.get('n_sessions', 0))} production session(s)")

---
## 10. Inspection -- TensorBoard + the cross-iteration dashboard

TRL writes one event file per iteration, each restarting at `global_step` 0, so the web UI shows
N disconnected curves. `plot_iteration_metrics` stitches them after the fact (per-iteration step
offsets, dotted lines at the boundaries); the optional `tb_live/` run (`TB_LIVE_LOGGING`) is the
during-training answer and is already on a cumulative axis -- which is why the dashboard skips it.

In [ ]:
import socket
from pathlib import Path

scan_scalar_tags(paths.run_dir)

with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as _s:
    _s.bind(("127.0.0.1", 0))
    tb_port = _s.getsockname()[1]

# Forward slashes: the IPython magic re-parses Windows backslashes as escape sequences.
tb_logdir = Path(paths.run_dir).as_posix()
print(f"TensorBoard on port {tb_port}  logdir {tb_logdir}")
print(f"If the inline iframe does not render (common in VS Code): http://localhost:{tb_port}")

%reload_ext tensorboard
%tensorboard --logdir "{tb_logdir}" --port {tb_port}

In [ ]:
%matplotlib inline

# Auto-detects DPO (this notebook) vs GRPO from the tags TRL actually wrote and plots that trainer's
# own metrics -- rewards/chosen, rewards/rejected, accuracies, margins, logps.
plot_iteration_metrics(paths.run_dir)

In [ ]:
# Optional: hand the server's pre-allocation back (VLLM_GPU_MEMORY_UTILIZATION x 40 GB). Only do
# this when nothing else needs the grader -- scoring (eda/notebooks/scoring/Run_Eval.ipynb) needs a
# server too, and stop() is a no-op on a handle this process merely ADOPTED, by design.
#
# for _handle in SERVER_HANDLES.values():
#     _handle.stop()
# print("vLLM servers stopped")

---
## What happens next

1. **Score.** `eda/notebooks/scoring/Run_Eval.ipynb` -- arms are auto-discovered from disk, so
   there is no registry to edit. Scores land in
   `data/eval_scores/judge=<tag>/rep=<r>/metric=<M>/<EXPERIMENT_NAME>/model_iter_<N>.parquet`, one
   parquet of 96 rows per model state.
2. **Analyse.** The family notebooks under `eda/notebooks/`, or `python tools/render_results.py`.

### If something went wrong

| symptom | what it actually is |
|---|---|
| `Iteration N has 0 preference pairs` | usually an EMPTY `pairs.csv` left by a killed process. **Delete it** and re-run. Never lower `PREF_FILTER_TAU` to get past it. |
| `stale _progress.json ... discarding it` | the snapshot was built under a different tau / M / MCL / trunk target / seed. Discarding is correct: half a build at one tau plus half at another is a folder nobody can interpret. |
| oracle success rate below the floor -> `RuntimeError` | the grader is failing on a large share of the batch. `_progress.json` survives, so fixing the server and re-running resumes the build rather than restarting it. |
| a conversation folder reads as empty | on Windows the Drive mount can wedge on a single folder (`WinError 1450`) while every file is present in the cloud. **Check the cloud before concluding an arm is unfinished.** |
| iteration wall-clock looks impossible | read `timing_sessions.jsonl`, not a per-process field. `n_sessions_production > 1` means the iteration was resumed and every per-process number for it undercounts. (Not `n_sessions`: the final-eval pass adds an eval-gen-only session to the last iteration of every healthy arm.) |
| the look-ahead sub-batch in the metadata is lower than cell 1's | an OOM halved it, and the halving is sticky. That is recorded per iteration precisely because it is not in the arm name, and per-iteration wall-clock stops being comparable without it. |

### Reminders

- `EXPERIMENT_NAME` is computed. If you want a different arm, change K, MCL, M, the tree mode, the
  rubric or a role model -- never the string.
- Everything the name does not encode lives only in `run_metadata.json` + its history log.
- PTO is the framework, DPO is the loss. GRPO has no preference data, only prompts.